In [ ]:
import numpy as np 
import math
import os
from tqdm.notebook import tqdm
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import matplotlib as mpl
import matplotlib.lines as mlines
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scipy.stats as sp
from scipy.stats import kendalltau
from pingouin import partial_corr

cmap = sns.color_palette("colorblind")


In [ ]:
def corr_datscan(crit, datscan, age): 
    rho = np.zeros((30, len(datscan)))
    p = np.zeros_like(rho)

    for i_dat, dat in enumerate(datscan): 

        # Check nan in datscan file
        idx_dat = np.argwhere(np.isnan(datscan))
        dats = np.delete(dat, idx_dat)
        crit_dats = np.delete(crit, idx_dat, axis=0)
        age_dats = np.delete(age, idx_dat, axis=0)
        
        # Spearman's correlation
        for iF in range(30):
            rho[iF, i_dat], p[iF, i_dat] = sp.spearmanr(crit_dats[:,iF], dats)
        
    # Benjiamini - Hochberg's correction
    _, p_corrected, _, _ = multipletests(p.ravel(), method='fdr_bh')

    p_corr = p_corrected.reshape([30,2])
       
    return rho, p, p_corr

def corr_datscan_single_Ch(crit, datscan, age): 
    _, nF = crit.shape
    rho = np.zeros((nF, len(datscan)))
    p = np.zeros_like(rho)

    for i_dat, dat in enumerate(datscan): 

        # Check nan in datscan file
        idx_dat = np.argwhere(np.isnan(datscan))
        dats = np.delete(dat, idx_dat)
        crit_dats = np.delete(crit, idx_dat, axis=0)
        age_dats = np.delete(age, idx_dat, axis=0)
        
        # Spearman's correlation
        for iF in range(crit_dats.shape[1]):
            rho[iF, i_dat], p[iF, i_dat] = sp.spearmanr(crit_dats[:,iF], dats)
             
    return rho, p


def freq_band_extraction(data1, data2, freq_range):
    data1 = data1[:,:,freq_range].mean(axis=2)
    data2 = data2[:,:,freq_range].mean(axis=2)
    data = np.concatenate([data1, data2])

    return data

def corr_clinical(eeg, score, freq_vals): 
    
    rho_score, p_score = [np.zeros((len(freq_vals),)) for _ in range(2)]

    for i, iF in enumerate(freq_vals): 
            
        rho_score[i], p_score[i] = kendalltau(eeg[:,i], score, nan_policy='omit')
            
    _, p_score_corr, _, _ = multipletests(p_score, method='fdr_bh')

    return rho_score, p_score, p_score_corr

def extract_bis_new(files):
    
    data, name_subjs = list(), list()
    
    for file in tqdm(files):
        
        name_subjs.append(file.split('/')[9][:6])
        
        tmp = np.load(file, allow_pickle=True)
        
        data.append(np.mean(tmp,axis=0)) # Average channels

    bis = np.array(data)

    return name_subjs, bis

def extract_dfa_new(files):
    
    dfa, name_subjs = list(), list()

    for file in tqdm(files):
        name_subjs.append(file.split('/')[9][:6])
        datafile = np.load(file, allow_pickle=True)
        data = datafile.item()
        data_dfa = data['DFA']
        dfa.append(np.nanmean(data_dfa,axis=0))

    dfa = np.array(dfa)

    return name_subjs, dfa


def extract_fei_new(files):
    
    name_subjs, data_masked = list(), list()
    
    for file in tqdm(files):    
        name_subjs.append(file.split('/')[9][:6])
        tmp = np.load(file, allow_pickle=True)
        data_masked.append(np.nanmean(tmp[1,...],axis=0)) # Average channels without nan
    
    fEI_masked = np.array(data_masked)

    return name_subjs, fEI_masked


#### Set input

In [ ]:
# Import LOOK UP TABLE - RBD 
lut_rbd = pd.read_excel('fname_rbd').set_index('ID')


# Set frequency limits
f_min = 2
f_max = 90
scale_freq='log'

#### Extract dopamine and BiS information

In [ ]:
# Extract age
age_r1 = lut_rbd.loc[(lut_rbd['REC']=='B')]['Age'].to_numpy()
age_r2 = lut_rbd.loc[(lut_rbd['REC']=='FU1')]['Age'].to_numpy()
age = np.concatenate((age_r1,age_r2))

In [ ]:
# Extract information about conveterted and not converted patients
name_lateRBD_r1 = lut_rbd.loc[(~lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'B') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 1) ].index.to_list()
name_lateRBD_r1 = [i.split('_', 1)[0] for i in name_lateRBD_r1]
name_lateRBD_r2 = lut_rbd.loc[(~lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'FU1') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 1)  ].index.to_list()
name_lateRBD_r2 = [i.split('_', 1)[0] for i in name_lateRBD_r2]

name_earlyRBD_r1 = lut_rbd.loc[(~lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'B') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 2) ].index.to_list()
name_earlyRBD_r1 = [i.split('_', 1)[0] for i in name_earlyRBD_r1]
name_earlyRBD_r2 = lut_rbd.loc[(~lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'FU1') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 2) ].index.to_list()
name_earlyRBD_r2 = [i.split('_', 1)[0] for i in name_earlyRBD_r2]

name_sRBD_r1 = lut_rbd.loc[(lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'B') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 0)].index.to_list()
name_sRBD_r1 = [i.split('_', 1)[0] for i in name_sRBD_r1]
name_sRBD_r2 = lut_rbd.loc[(lut_rbd['CONVERSION TO '].isnull()) & (lut_rbd['REC'] == 'FU1') & (lut_rbd['CONV (0=non, 1=late, 2=early)'] == 0)].index.to_list()
name_sRBD_r2 = [i.split('_', 1)[0] for i in name_sRBD_r2]

In [ ]:
# Extract information about dopamine levels 
caudL_r1 = lut_rbd['Caudate left'].loc[lut_rbd['REC']=='B'].to_numpy()
caudR_r1 = lut_rbd['Caudate right'].loc[lut_rbd['REC']=='B'].to_numpy()

putL_r1 = lut_rbd['Putamen left'].loc[lut_rbd['REC']=='B'].to_numpy()
putR_r1 = lut_rbd['Putamen right'].loc[lut_rbd['REC']=='B'].to_numpy()

dat_r1 = [(putL_r1 + putR_r1)/2, (caudL_r1 + caudR_r1)/2]

caudL_r2 = lut_rbd['Caudate left'].loc[lut_rbd['REC']=='FU1'].to_numpy()
caudR_r2 = lut_rbd['Caudate right'].loc[lut_rbd['REC']=='FU1'].to_numpy()

putL_r2 = lut_rbd['Putamen left'].loc[lut_rbd['REC']=='FU1'].to_numpy()
putR_r2 = lut_rbd['Putamen right'].loc[lut_rbd['REC']=='FU1'].to_numpy()

dat_r2 = [(putL_r2 + putR_r2)/2, (caudL_r2 + caudR_r2)/2]

datscan = np.hstack((dat_r1, dat_r2))

In [ ]:
# MMSE
id_r1 = lut_rbd.loc[(lut_rbd['REC'] == 'B')].index
mmse_r1 = lut_rbd.loc[id_r1, 'MMSE'].to_numpy()

id_r2 = lut_rbd.loc[(lut_rbd['REC'] == 'FU1')].index
mmse_r2 = lut_rbd.loc[id_r2, 'MMSE'].to_numpy()

mmse = np.concatenate((mmse_r1, mmse_r2))

# UPDRS-III
id_r1 = lut_rbd.loc[(lut_rbd['REC'] == 'B')].index
updrs_r1 = lut_rbd.loc[id_r1, 'UPDRS-III'].to_numpy()

id_r2 = lut_rbd.loc[(lut_rbd['REC'] == 'FU1')].index
updrs_r2 = lut_rbd.loc[id_r2, 'UPDRS-III'].to_numpy()

updrs = np.concatenate((updrs_r1, updrs_r2))

In [ ]:
name_subj_r1, fei_r1 = extract_fei_new(sorted(glob.glob(os.path.join('fei_r1'))))
name_subj_r2, fei_r2 = extract_fei_new(sorted(glob.glob(os.path.join('fei_r2'))))

_, bis_r1 = extract_bis_new(sorted(glob.glob(os.path.join('bis_r1'))))
_, bis_r2 = extract_bis_new(sorted(glob.glob(os.path.join('bis_r2'))))

_, dfa_r1 = extract_dfa_new(sorted(glob.glob(os.path.join('dfa_r1'))))
_, dfa_r2 = extract_dfa_new(sorted(glob.glob(os.path.join('dfa_r2'))))


In [ ]:
# Set frequency axis
def create_frequency_axis(f_min=2, f_max= 4, scale_freq ='log', m=1.1):
    
    if scale_freq == 'log':
        f_min_log = math.floor(np.log10(f_min))
        f_max_log = math.ceil(np.log10(f_max))
        mb = np.logspace(f_min_log, f_max_log, num=40) # Morlet bank 
        freq_vals = mb[(2.1 < mb) & (mb < 90)] # Frequency of interest
        
    
    elif scale_freq == 'lin':
        freq_vals = [f_min]
        while freq_vals[~0] < f_max:
            freq_vals.append(freq_vals[~0]*m)

    freq_labels = [('%.2f'%x) for x in freq_vals]
    
    return freq_vals, freq_labels


freq_vals, _ = create_frequency_axis(f_min=f_min, f_max=f_max, scale_freq =scale_freq)

In [ ]:
freq_vals = freq_vals[:30]

bis = np.concatenate((bis_r1[:,:30],bis_r2[:,:30]))
fei = np.concatenate((fei_r1[:,:30],fei_r2[:,:30]))
dfa = np.concatenate((dfa_r1[:,:30],dfa_r2[:,:30]))

#### Correlation dopamine levels vs BiS

In [ ]:
rho_fei, p_fei, p_fei_corrected = corr_datscan(fei, datscan, age)

rho_bis, p_bis, p_bis_corrected = corr_datscan(bis, datscan, age)

rho_dfa, p_dfa, p_dfa_corrected = corr_datscan(dfa, datscan, age)

rho_fei_put_lst, rho_bis_put_lst, rho_dfa_put_lst, rho_fei_cau_lst, rho_bis_cau_lst, rho_dfa_cau_lst = list(), list(), list(), list(), list(), list()

for iF, _ in enumerate(freq_vals): 
    ddf = pd.DataFrame()
    ddf['Putamen'] = datscan[0]
    ddf['Caudate'] = datscan[1]
    ddf['Age'] = age
    ddf['SEX'] = df['GENDER'].values
    ddf['fEI'] = fei[:, iF]
    ddf['BiS'] = bis[:, iF]
    ddf['DFA'] = dfa[:, iF]

    ddf.dropna(inplace=True)
 
    stats_fei_putamen = partial_corr(data=ddf, x='fEI', y='Putamen', covar='Age', method='spearman')
    stats_bis_putamen = partial_corr(data=ddf, x='BiS', y='Putamen', covar='Age', method='spearman')
    stats_dfa_putamen = partial_corr(data=ddf, x='DFA', y='Putamen', covar='Age', method='spearman')

    stats_fei_caudate = partial_corr(data=ddf, x='fEI', y='Caudate', covar='Age', method='spearman')
    stats_bis_caudate = partial_corr(data=ddf, x='BiS', y='Caudate', covar='Age', method='spearman')
    stats_dfa_caudate = partial_corr(data=ddf, x='DFA', y='Caudate', covar='Age', method='spearman')

    rho_fei_put_lst.append(stats_fei_putamen['r']) 
    rho_fei_cau_lst.append(stats_fei_caudate['r'])
    rho_bis_put_lst.append(stats_bis_putamen['r']) 
    rho_bis_cau_lst.append(stats_bis_caudate['r'])
    rho_dfa_put_lst.append(stats_dfa_putamen['r']) 
    rho_dfa_cau_lst.append(stats_dfa_caudate['r'])

rho_fei = np.concatenate((np.array(rho_fei_put_lst), np.array(rho_fei_cau_lst)), axis=1)
rho_bis = np.concatenate((np.array(rho_bis_put_lst), np.array(rho_bis_cau_lst)), axis=1)
rho_dfa = np.concatenate((np.array(rho_dfa_put_lst), np.array(rho_dfa_cau_lst)), axis=1)

#### Correlation with clinical scores

In [ ]:
#Correlazione bis vs updrs-iii
rho_updrs_bis, p_updrs_bis, p_corrected_updrs_bis = corr_clinical(bis, updrs, freq_vals)   
rho_updrs_fei, p_updrs_fei, p_corrected_updrs_fei = corr_clinical(fei, updrs, freq_vals)   
rho_updrs_dfa, p_updrs_dfa, p_corrected_updrs_dfa = corr_clinical(dfa, updrs, freq_vals)   

#Correlazione bis vs mmse
rho_mmse_bis, p_mmse_bis, p_corrected_mmse_bis = corr_clinical(bis, mmse, freq_vals) 
rho_mmse_fei, p_mmse_fei, p_corrected_mmse_fei = corr_clinical(fei, mmse, freq_vals) 
rho_mmse_dfa, p_mmse_dfa, p_corrected_mmse_dfa = corr_clinical(dfa, mmse, freq_vals) 

#### Index converted - not converted

In [ ]:
idx_r1_lateRBD = [name_subj_r1.index(x) for x in name_lateRBD_r1 if x in name_subj_r1]
idx_r2_lateRBD = [name_subj_r2.index(x) for x in name_lateRBD_r2 if x in name_subj_r2]

idx_r1_earlyRBD = [name_subj_r1.index(x) for x in name_earlyRBD_r2 if x in name_subj_r1]
idx_r2_earlyRBD = [name_subj_r2.index(x) for x in name_earlyRBD_r2 if x in name_subj_r2]

idx_r1_sRBD = [name_subj_r1.index(x) for x in name_sRBD_r1 if x in name_subj_r1]
idx_r2_sRBD = [name_subj_r2.index(x) for x in name_sRBD_r2 if x in name_subj_r2]

In [ ]:
idx_r1r2 = [name_subj_r1.index(x) for x in name_subj_r2 if x in name_subj_r1]

#### Figure 2

In [ ]:
# Create legend
handles = list()
handles.append(mlines.Line2D([], [], color=cmap[6], label='Putamen'))
handles.append(mlines.Line2D([], [], color=cmap[9], label='Caudate'))

handles.append(mlines.Line2D([], [], color='black', marker='o', linestyle='none',
                        markersize=10, label='pvalue < 0.05',markerfacecolor='none'))

handles.append(mlines.Line2D([], [], color='black', marker='o', linestyle='none',
                        markersize=10, label='pvalue corrected < 0.05'))

In [ ]:
def cc_plot(ax, freq_vals, rho, column, colors, eeg_label):

    a_size=8
    l_size=10
    
    for i_col in range(len(column)): 
        ax.semilogx(freq_vals, rho[:,i_col], color=colors[i_col])

    ax.tick_params(labelsize=a_size)
    ax.set_ylim([-.5, .5])
    ax.set_xticks([2,5,10,20,50,70], [2,5,10,20,50,70])
    ax.set_ylabel(f"CC {eeg_label} vs SBR", fontsize=l_size)
    ax.set_xlabel('Frequencies [Hz]', fontsize=l_size)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    

def scatter_plot(ax, df_base, df_fup, x_df, idx_r1_sRBD, idx_r1_lateRBD, idx_r1_earlyRBD,  eeg_r1, eeg_r2, dat_r1, dat_r2, cmap):
    
    m_size=8
    a_size=8
    l_size=10

    sns.regplot(x=x_df, y='Putamen', data=df_base, fit_reg=True, ci=95, n_boot=1000, 
            scatter_kws={'color':'white', 's':m_size}, line_kws={'color': 'darkgray'}, ax=ax)
    
    sns.regplot(x=x_df, y='Putamen', data=df_fup, fit_reg=True, ci=95, n_boot=1000, 
            scatter_kws={'color':'white', 's':m_size}, line_kws={'color': 'dimgray'}, ax=ax)


    # sRBD
    ax.scatter(eeg_r1[idx_r1_sRBD], 
                dat_r1[0][idx_r1_sRBD], marker='o', color=cmap[2], 
                label='nc-iRBD', s=m_size)
    ax.scatter(eeg_r2[idx_r2_sRBD], 
                dat_r2[0][idx_r2_sRBD], marker='o', color=cmap[2], s=m_size)

    # late RBD 
    ax.scatter(eeg_r1[idx_r1_lateRBD], 
                dat_r1[0][idx_r1_lateRBD], marker='^', 
                color=cmap[1], label='late-iRBD', s=m_size) 
    ax.scatter(eeg_r2[idx_r2_lateRBD], 
                dat_r2[0][idx_r2_lateRBD], marker='^', 
                color=cmap[1], s=m_size) 

    # early RBD
    ax.scatter(eeg_r1[idx_r1_earlyRBD], 
                dat_r1[0][idx_r1_earlyRBD], marker='X', color='red', 
                label='early-iRBD', s=m_size)
    ax.scatter(eeg_r2[idx_r2_earlyRBD], 
                dat_r2[0][idx_r2_earlyRBD], marker='X', color='red', 
                s=m_size) 

    ax.tick_params(labelsize=a_size)
    ax.set_ylabel('SBR - putamen', fontsize=l_size)
    ax.set_xlabel(x_df, fontsize=l_size)


    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

In [ ]:
import seaborn as sns
import pandas as pd

fig = plt.figure(figsize=(17/2.54, 13/2.54), layout='constrained')

gs = fig.add_gridspec(nrows=3, ncols=3)

ax1 = fig.add_subplot(gs[0, 0])
ax4 = fig.add_subplot(gs[0, 1])
ax7 = fig.add_subplot(gs[0, 2])

ax2 = fig.add_subplot(gs[1, 0])
ax5 = fig.add_subplot(gs[1, 1])
ax8 = fig.add_subplot(gs[1, 2])

ax3 = fig.add_subplot(gs[2, 0])
ax6 = fig.add_subplot(gs[2, 1])
ax9 = fig.add_subplot(gs[2, 2])

# Set input for plot
column = ['Putamen', 'Caudate']
colors = [cmap[6], cmap[9]]
i = [0.02, -0.02]
rho = [rho_dfa, rho_bis, rho_fei]
labels = ['DFA', 'BiS', 'fEI']

for i, ax in enumerate([ax1, ax2, ax3]):
    cc_plot(ax, freq_vals, rho[i], column, colors, labels[i])


nS_base = len(name_subj_r1)

df_base = pd.DataFrame({'DFA (8-13) Hz':dfa[:nS_base,11:16].mean(axis=1),'BiS (2-4) Hz':bis[:nS_base,:7].mean(axis=1), 'fEI (5-7) Hz':fei[:nS_base,7:11].mean(axis=1),
                   'Putamen':datscan[0][:nS_base]})

df_fup = pd.DataFrame({'DFA (8-13) Hz':dfa[nS_base:,11:16].mean(axis=1),'BiS (2-4) Hz':bis[nS_base:,:7].mean(axis=1), 'fEI (5-7) Hz':fei[nS_base:,7:11].mean(axis=1),
                   'Putamen':datscan[0][nS_base:]})

x_df = ['DFA (8-13) Hz', 'BiS (2-4) Hz', 'fEI (5-7) Hz']
freqs_range = [11,16,7,11,]

eeg_r1 = [dfa_r1[:,11:16].mean(axis=1), bis_r1[:,7:11].mean(axis=1), fei_r1[:,7:11].mean(axis=1)]
eeg_r2 = [dfa_r2[:,11:16].mean(axis=1), bis_r2[:,7:11].mean(axis=1), fei_r2[:,7:11].mean(axis=1)]

for i, ax in enumerate([ax4, ax5, ax6]):
    scatter_plot(ax, df_base, df_fup, x_df[i], idx_r1_sRBD, idx_r1_lateRBD, idx_r1_earlyRBD, eeg_r1[i], eeg_r2[i], dat_r1, dat_r2, cmap)

column_cs = ['MDS-UPDRS-III', 'MMSE']
colors_cs = [cmap[5], cmap[7]]
rho_cs = [np.concatenate((rho_updrs_dfa[np.newaxis], rho_mmse_dfa[np.newaxis]),axis=0).T, 
          np.concatenate((rho_updrs_bis[np.newaxis], rho_mmse_bis[np.newaxis]), axis=0).T, 
          np.concatenate((rho_updrs_fei[np.newaxis], rho_mmse_fei[np.newaxis]),axis=0).T]

for i, ax in enumerate([ax7,ax8,ax9]):
    cc_plot(ax, freq_vals, rho_cs[i], column_cs, colors_cs, labels[i])



In [ ]:
def split_freq_range(eeg_measure):
    
    eeg_delta = eeg_measure[:, :7].mean(axis=1)
    eeg_theta = eeg_measure[:, 7:11].mean(axis=1)
    eeg_alpha = eeg_measure[:, 11:16].mean(axis=1)
    eeg_beta = eeg_measure[:, 16:22].mean(axis=1)
    eeg_gamma = eeg_measure[:, 22:30].mean(axis=1)

    return eeg_delta, eeg_theta, eeg_alpha, eeg_beta, eeg_gamma



In [ ]:
dfa_delta, dfa_theta, dfa_alpha, dfa_beta, dfa_gamma = split_freq_range(dfa)
bis_delta, bis_theta, bis_alpha, bis_beta, bis_gamma = split_freq_range(bis)
fei_delta, fei_theta, fei_alpha, fei_beta, fei_gamma = split_freq_range(fei)

In [ ]:
df = pd.read_csv('fname_rbd').set_index('Unnamed: 0')
df.loc[df['Sex']==0, 'Sex'] = 'M'
df.loc[df['Sex']==1, 'Sex'] = 'F'

In [ ]:
from sklearn.preprocessing import StandardScaler

ddf = pd.DataFrame()

ddf['DFAdelta'] = dfa_delta
ddf['DFAtheta'] = dfa_theta
ddf['DFAalpha'] = dfa_alpha
ddf['DFAbeta'] = dfa_beta
ddf['DFAgamma'] = dfa_gamma

ddf['BiSdelta'] = bis_delta
ddf['BiStheta'] = bis_theta
ddf['BiSalpha'] = bis_alpha
ddf['BiSbeta'] = bis_beta
ddf['BiSgamma'] = bis_gamma

ddf['fEIdelta'] = fei_delta
ddf['fEItheta'] = fei_theta
ddf['fEIalpha'] = fei_alpha
ddf['fEIbeta'] = fei_beta
ddf['fEIgamma'] = fei_gamma
ddf['SBRPutamen'] = datscan[0]
ddf['MMSE'] = mmse
ddf['UPDRS'] = updrs

scaler = StandardScaler()
df_standardized = pd.DataFrame(scaler.fit_transform(ddf.iloc[:,:19]), columns=ddf.columns[:19])

df_standardized['Sex'] = df['Sex'].tolist()
df_standardized['Age'] = age
df_standardized['Subjects'] = df['Subjects'].tolist()


In [ ]:
from statsmodels.formula.api import mixedlm

dep_var = ['SBRPutamen', 'MMSE', 'UPDRS']

for ivar in dep_var:
    formula = f"{ivar} ~ DFAdelta + DFAtheta + DFAalpha + DFAbeta + DFAgamma + Age + Sex"

    ddf_standardized = df_standardized[['DFAdelta', 'DFAtheta', 'DFAalpha', 'DFAbeta', 'DFAgamma', 'Age', 'Sex', 'Subjects', ivar]]

    ddf_standardized.dropna(inplace=True)

    model = mixedlm(formula, ddf_standardized, groups=ddf_standardized["Subjects"]).fit()

    fitted_model = model.summary().tables[1]
    print(fitted_model)


In [ ]:
for ivar in dep_var:
    formula = f"{ivar} ~ BiSdelta + BiStheta + BiSalpha + BiSbeta + BiSgamma + Age + Sex"

    ddf_standardized = df_standardized[['BiSdelta', 'BiStheta', 'BiSalpha', 'BiSbeta', 'BiSgamma', 'Age', 'Sex', 'Subjects', ivar]]

    ddf_standardized.dropna(inplace=True)

    model = mixedlm(formula, ddf_standardized, groups=ddf_standardized["Subjects"]).fit()

    fitted_model = model.summary().tables[1]
    print(fitted_model)


In [ ]:
for ivar in dep_var:
    formula = f"{ivar} ~ fEIdelta + fEItheta + fEIalpha + fEIbeta + fEIgamma + Age + Sex"

    ddf_standardized = df_standardized[['fEIdelta', 'fEItheta', 'fEIalpha', 'fEIbeta', 'fEIgamma', 'Age', 'Sex', 'Subjects', ivar]]

    ddf_standardized.dropna(inplace=True)

    model = mixedlm(formula, ddf_standardized, groups=ddf_standardized["Subjects"]).fit()

    fitted_model = model.summary().tables[1]

    print(fitted_model)